## load dataset

In [26]:
import numpy as np
import math
from math import prod
from decimal import Decimal, getcontext
import pickle
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import json
import os
getcontext().prec = 5

In [27]:
import pandas as pd
import numpy as np

# Muat dataset
column_names = ['user_id', 'item_id', 'rating', 'timestamp']
df = pd.read_csv('../../Dataset/ml-100k/u.data', sep='\t', names=column_names)

# Buat matriks peringkat pengguna-item
n_users = df.user_id.nunique()
n_items = df.item_id.nunique()
ratings = np.zeros((n_users, n_items))
for row in df.itertuples():
    ratings[row[1]-1, row[2]-1] = row[3]

print("Bentuk matriks peringkat:", ratings.shape)
print("Contoh matriks peringkat:")
print("jumlah rating:", np.count_nonzero(ratings))
print(ratings[:10, :10])


Bentuk matriks peringkat: (943, 1682)
Contoh matriks peringkat:
jumlah rating: 100000
[[5. 3. 4. 3. 3. 5. 4. 1. 5. 3.]
 [4. 0. 0. 0. 0. 0. 0. 0. 0. 2.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [4. 3. 0. 0. 0. 0. 0. 0. 0. 0.]
 [4. 0. 0. 0. 0. 0. 2. 4. 4. 0.]
 [0. 0. 0. 5. 0. 0. 5. 5. 5. 4.]
 [0. 0. 0. 0. 0. 0. 3. 0. 0. 0.]
 [0. 0. 0. 0. 0. 5. 4. 0. 0. 0.]
 [4. 0. 0. 4. 0. 0. 4. 0. 4. 0.]]


## mecari mean rating dan mean centered rating

In [28]:
def mean_rating_user(ratings):  # Menghitung mean rating user
    non_zero_ratings = np.ma.masked_equal(ratings, 0)
    return np.mean(non_zero_ratings, axis=1)

def mean_rating_item(ratings):  # Menghitung mean rating item
    non_zero_ratings = np.ma.masked_equal(ratings, 0)
    return np.mean(non_zero_ratings, axis=0)

def mean_centered_rating_user(ratings):  # Mean centering berdasarkan user
    mean_ratings = mean_rating_user(ratings)
    non_zero_ratings = np.ma.masked_equal(ratings, 0)  # masking rating 0
    centered_ratings = non_zero_ratings - mean_ratings[:, np.newaxis]
    return centered_ratings.filled(0)

def mean_centered_rating_item(ratings):  # Mean centering berdasarkan item
    mean_ratings = mean_rating_item(ratings)
    non_zero_ratings = np.ma.masked_equal(ratings, 0)  # masking rating 0
    centered_ratings = non_zero_ratings - mean_ratings
    return centered_ratings.filled(0)


In [29]:
mru = mean_rating_user(ratings)

In [30]:
print(mru)

[3.610294117647059 3.7096774193548385 2.7962962962962963 4.333333333333333
 2.874285714285714 3.6350710900473935 3.965260545905707 3.7966101694915255
 4.2727272727272725 4.206521739130435 3.4640883977900554 4.392156862745098
 3.09748427672956 4.091836734693878 2.875 4.328571428571428
 3.0357142857142856 3.88086642599278 3.55 3.1041666666666665
 2.670391061452514 3.3515625 3.6357615894039736 4.323529411764706
 4.051282051282051 2.94392523364486 3.24 3.721518987341772
 3.6470588235294117 3.7674418604651163 3.9166666666666665
 3.317073170731707 3.7083333333333335 4.05 3.0 3.8 3.6491228070175437
 3.71900826446281 3.772727272727273 2.8857142857142857 3.769230769230769
 3.726775956284153 3.7104072398190047 3.6490066225165565
 3.6041666666666665 4.111111111111111 3.64 3.787878787878788
 2.683720930232558 3.5416666666666665 3.5652173913043477 4.267857142857143
 3.8214285714285716 3.6923076923076925 3.2857142857142856
 3.641711229946524 3.5 3.8376623376623376 3.9345549738219896
 4.0865384615384

In [31]:
mcru = mean_centered_rating_user(ratings)
print("Bentuk matriks mean-centered rating user:", mcru.shape)
print("Contoh matriks mean-centered rating user:")
print(mcru[:10, :10])

Bentuk matriks mean-centered rating user: (943, 1682)
Contoh matriks mean-centered rating user:
[[ 1.38970588 -0.61029412  0.38970588 -0.61029412 -0.61029412  1.38970588
   0.38970588 -2.61029412  1.38970588 -0.61029412]
 [ 0.29032258  0.          0.          0.          0.          0.
   0.          0.          0.         -1.70967742]
 [ 0.          0.          0.          0.          0.          0.
   0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.          0.          0.
   0.          0.          0.          0.        ]
 [ 1.12571429  0.12571429  0.          0.          0.          0.
   0.          0.          0.          0.        ]
 [ 0.36492891  0.          0.          0.          0.          0.
  -1.63507109  0.36492891  0.36492891  0.        ]
 [ 0.          0.          0.          1.03473945  0.          0.
   1.03473945  1.03473945  1.03473945  0.03473945]
 [ 0.          0.          0.          0.          0.          0.
  -0.796610

In [32]:
mcri = mean_centered_rating_item(ratings)
print("Bentuk matriks mean-centered rating item:", mcri.shape)
print("Contoh matriks mean-centered rating item:")
print(mcri[:10, :10])

Bentuk matriks mean-centered rating item: (943, 1682)
Contoh matriks mean-centered rating item:
[[ 1.12168142 -0.20610687  0.96666667 -0.55023923 -0.30232558  1.42307692
   0.20153061 -2.99543379  1.10367893 -0.83146067]
 [ 0.12168142  0.          0.          0.          0.          0.
   0.          0.          0.         -1.83146067]
 [ 0.          0.          0.          0.          0.          0.
   0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.          0.          0.
   0.          0.          0.          0.        ]
 [ 0.12168142 -0.20610687  0.          0.          0.          0.
   0.          0.          0.          0.        ]
 [ 0.12168142  0.          0.          0.          0.          0.
  -1.79846939  0.00456621  0.10367893  0.        ]
 [ 0.          0.          0.          1.44976077  0.          0.
   1.20153061  1.00456621  1.10367893  0.16853933]
 [ 0.          0.          0.          0.          0.          0.
  -0.798469

## PCC


In [33]:
from itertools import combinations

def pcc(ratings, base='user'):
    if base == 'user':
        mean_centered_rating = mean_centered_rating_user(ratings)
        num_users = mean_centered_rating.shape[0]  # Jumlah user
        # print(num_users)
        pcc_matrix_users = np.eye(num_users)   # Inisialisasi matriks PCC untuk user
        user_masks = [ratings[row, :] != 0 for row in range(num_users)]  # masking rating 0

        for u, v in combinations(range(num_users), 2):
            common_items = user_masks[u] & user_masks[v]
            if np.sum(common_items) == 0:
                continue
            
            u_common_ratings = mean_centered_rating[u, common_items]  # penyebut kiri
            v_common_ratings = mean_centered_rating[v, common_items]  # penyebut kanan

            numerator = np.sum(u_common_ratings * v_common_ratings)
            denominator = np.sqrt(np.sum(u_common_ratings**2)) * np.sqrt(np.sum(v_common_ratings**2))
            
            if denominator != 0:
                pcc_value = numerator / denominator
            else:
                pcc_value = 0
            
            pcc_matrix_users[u, v] = pcc_value
            pcc_matrix_users[v, u] = pcc_value
        return pcc_matrix_users
    
    elif base == 'item':
        mean_centered_ratings = mean_centered_rating_item(ratings)
        num_items = mean_centered_ratings.shape[1] # jumlah item
        pcc_matrix_item = np.eye(num_items)  # Inisialisasi matriks PCC untuk item
        item_masks = [ratings[:, col] != 0 for col in range(num_items)]  # masking rating 0

        for i, j in combinations(range(num_items), 2):
            common_users = item_masks[i] & item_masks[j]
            if np.sum(common_users) == 0:
                continue

            i_common_ratings = mean_centered_ratings[common_users, i]
            j_common_ratings = mean_centered_ratings[common_users, j]

            numerator = np.sum(i_common_ratings * j_common_ratings)
            denominator = np.sqrt(np.sum(i_common_ratings**2)) * np.sqrt(np.sum(j_common_ratings**2))
            
            if denominator != 0:
                pcc_value = numerator / denominator
            else:
                pcc_value = 0
            
            pcc_matrix_item[i, j] = pcc_value
            pcc_matrix_item[j, i] = pcc_value
        return pcc_matrix_item
    
    else:
        raise ValueError("Base must be either 'user' or 'item'.")


In [34]:
pcc_matrix_users = pcc(ratings, base='user')
pcc_matrix_items = pcc(ratings, base='item')

In [35]:
pcc_matrix_users

array([[ 1.        ,  0.36087079,  0.15722789, ...,  0.26729899,
        -0.16403   ,  0.1078816 ],
       [ 0.36087079,  1.        ,  0.13955603, ..., -0.1140556 ,
         0.05400834,  0.61074644],
       [ 0.15722789,  0.13955603,  1.        , ...,  0.71724476,
        -0.05553127,  0.17579064],
       ...,
       [ 0.26729899, -0.1140556 ,  0.71724476, ...,  1.        ,
        -0.45148352,  0.01180366],
       [-0.16403   ,  0.05400834, -0.05553127, ..., -0.45148352,
         1.        ,  0.4230873 ],
       [ 0.1078816 ,  0.61074644,  0.17579064, ...,  0.01180366,
         0.4230873 ,  1.        ]], shape=(943, 943))

In [36]:
pcc_matrix_items

array([[1.        , 0.22182379, 0.18626041, ..., 0.        , 0.        ,
        0.        ],
       [0.22182379, 1.        , 0.2339588 , ..., 0.        , 0.        ,
        0.        ],
       [0.18626041, 0.2339588 , 1.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 1.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 1.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        1.        ]], shape=(1682, 1682))

## mencari tetangga

In [37]:
def tuku_user(rating, pcc_matrix_users, user_id, item_id, k=20):
    b_user = rating.shape[0]
    user_similarities = pcc_matrix_users[user_id, :]
    # print(user_similarities)
    tuku = []
    for v in range(b_user):
        if v != user_id and rating[v, item_id] != 0:
            tuku.append((user_similarities[v], v))
    tuku.sort(key=lambda x: (-x[0], x[1]))
    return [n for n in tuku if n[0] > 0][:k]
def tuku_item(rating, pcc_matrix_items, user_id, item_id, k=20):
    b_item = rating.shape[1]
    item_similarities = pcc_matrix_items[item_id, :]
    tuku = []
    for j in range(b_item):
        if j != item_id and rating[user_id, j] != 0:
            tuku.append((item_similarities[j], j))
    tuku.sort(key=lambda x: (-x[0], x[1]))
    return [n for n in tuku if n[0] > 0][:k]


In [38]:
rating = ratings
rating1 = ratings[1]
rating
rating1

array([4., 0., 0., ..., 0., 0., 0.], shape=(1682,))

In [39]:
tetangga = tuku_user(ratings, pcc_matrix_users, user_id=0, item_id=0, k=4)

In [40]:
tetangga

[(np.float64(0.8233552316574889), 690),
 (np.float64(0.7644121379556182), 133),
 (np.float64(0.7450582773777711), 894),
 (np.float64(0.7440070677182473), 549)]

## mencari nilai probabilitas Prior

In [41]:
def prior_prob(rating, target_user=None, target_item=None, alpha=0.01, R=5):
    b_user = len(rating)
    b_item = len(rating[0])
    kelas_rating = list(range(1, R+1))

    if target_user is None:
        target_user = list(range(b_user))
        print("target user", target_user)
    if target_item is None:
        target_item = list(range(b_item))
        print("target item", target_item)

    prior_ub = np.zeros((R, len(target_item)))  #
    prior_ib = np.zeros((R, len(target_user)))

    item_map_prior = {item_idx: i for i, item_idx in enumerate(target_item)}
    user_map_prior = {user_idx: i for i, user_idx in enumerate(target_user)}

    for y in kelas_rating:
        y_index = y - 1

        # prior user-based
        for j in target_item:
            rating_item = rating[:, j]
            non_zero_ratings = rating_item[rating_item != 0]
            len_nonzero = len(non_zero_ratings)
            val = 0.0
            count_y = np.sum(non_zero_ratings == y)  # menghitung jumlah rating yang sama dengan y untuk item j

            if len_nonzero > 0:
                val = (count_y + alpha) / (len_nonzero + alpha * R)
            else:
                val = alpha / (alpha * R)
            prior_ub[y_index][item_map_prior[j]] = val
        
        # prior item-based
        for u in target_user:
            rating_user = rating[u, :]
            non_zero_ratings = rating_user[rating_user != 0]
            len_nonzero = len(non_zero_ratings)
            count_y = np.sum(non_zero_ratings == y)
            
            if len_nonzero > 0:
                val = (count_y + alpha) / (len_nonzero + alpha * R)
            else:
                val = alpha / (alpha * R)
            prior_ib[y_index][user_map_prior[u]] = val
    return prior_ub, prior_ib

In [42]:
t_user = tuku_user(rating, pcc_matrix_users, user_id=2, item_id=3, k=4)
print(t_user)
t_item = tuku_item(rating, pcc_matrix_items, user_id=2, item_id=3, k=1)
print(t_item)

[(np.float64(1.0), 217), (np.float64(1.0), 232), (np.float64(1.0), 397), (np.float64(1.0), 599)]
[(np.float64(0.6234969165040215), 352)]


## mencari nilai likelihood dengan tetangga

In [43]:
from numba import njit

tetangga_user = np.array([n[1] for n in t_user], dtype=np.int32)
tetangga_item = np.array([n[1] for n in t_item], dtype=np.int32)

@njit
def compute_likelihood_user(rating, u, i, y, tetangga_item, alpha=0.01, R=5):
    if len(tetangga_item) == 0:
        return 1.0
    product = 1.0
    u_with_y = np.where(rating[:, i] == y)[0]  # mengetahui r_v,i = y

    for j in tetangga_item:
        k = rating[u, j]  # r_u,j != 0 mengetahui r_u,i = y
        kondisi_pembilang = 0.0
        kondisi_penyebut = 0.0
        for v in u_with_y:
            if rating[v, j] != 0:
                kondisi_penyebut += 1.0
                if rating[v, j] == k:
                    kondisi_pembilang += 1.0
        probabilitas = (kondisi_pembilang + alpha) / (kondisi_penyebut + (R * alpha))
        product *= probabilitas
    return product

@njit
def compute_likelihood_item(rating, u, i, y, tetangga_user, alpha=0.01, R=5):
    if len(tetangga_user) == 0:
        return 1.0
    product = 1.0
    i_with_y = np.where(rating[u, :] == y)[0]

    for v in tetangga_user:
        k = rating[v, i]
        kondisi_pembilang = 0.0
        kondisi_penyebut = 0.0
        for j in i_with_y:
            if rating[v, j] != 0:
                kondisi_penyebut += 1.0
                if rating[v, j] == k:
                    kondisi_pembilang += 1.0
        probabilitas = (kondisi_pembilang + alpha) / (kondisi_penyebut + (R * alpha))
        product *= probabilitas
    return product




## prediksi rating hybrid

In [44]:
def prediksi_rating(rating, user, item, user_map, item_map, prior_ub, prior_ib, pcc_matrix_users, pcc_matrix_items, prior_u_map=None, prior_i_map=None, k_user=20, k_item=20, R=5):
    user_global = user - 1  
    item_global = item - 1

    t_user = tuku_user(rating, pcc_matrix_users, user_global, item_global, k_user)
    t_item = tuku_item(rating, pcc_matrix_items, user_global, item_global, k_item)

    tetangga_ub = np.array([n[1] for n in t_user], dtype=np.int32)
    tetangga_ib = np.array([n[1] for n in t_item], dtype=np.int32)

    user_dense = user_map.get(user)
    item_dense = item_map.get(item)

    if user_dense is None or item_dense is None:
        return None, [], []
    
    if prior_u_map is not None:
        if item_dense in prior_u_map:
            idx_prior_user = prior_u_map[item_dense]
        else:
            return None, [], []
    
    if prior_i_map is not None:
        if user_dense in prior_i_map:
            idx_prior_item = prior_i_map[user_dense]
        else:
            return None, [], []

    score_hybrid = []
    detail_y = []

    for y in range(1, R+1):
        prior_u = prior_ub[y-1][idx_prior_user]
        prior_i = prior_ib[y-1][idx_prior_item]

        li_u = compute_likelihood_user(rating, user_global, item_global, y, tetangga_ib, alpha=0.01, R=5)
        li_i = compute_likelihood_item(rating, user_global, item_global, y, tetangga_ub, alpha=0.01, R=5)

        posterior_ub = prior_u * li_u
        posterior_ib = prior_i * li_i

        score_ub = (posterior_ub) ** (1 / (k_item + 1)) if k_item > 0 else -100
        score_ib = (posterior_ib) ** (1 / (k_user + 1)) if k_user > 0 else -100

        hybrid_score = score_ub * score_ib
        score_hybrid.append(hybrid_score)
        detail_y.append({
            'rating_value' : y,
            'prior_user' : prior_u,
            'prior_item' : prior_i,
            'likelihood_user' : li_u,
            'likelihood_item' : li_i,
            'posterior_user' : posterior_ub,
            'posterior_item' : posterior_ib,
            'score_user' : score_ub,
            'score_item' : score_ib,
            'hybrid_score' : hybrid_score
        })
        
    max_score = max(score_hybrid)
    idx = [i for i, score in enumerate(score_hybrid) if np.isclose(score, max_score)]

    if len(idx) == 1:
        prediksi_rating = idx[0] + 1
    elif len(idx) > 1:
        Z = sum(score_hybrid)
        product = 0.0
        for idx_rating, score in enumerate(score_hybrid, start=1):
            normalisasi = score / Z
            prob = normalisasi * idx_rating
            product += prob
            prediksi_rating = product
    return prediksi_rating, score_hybrid, detail_y

In [45]:
# import numpy as np
# pcc = np.load("pcc_matrice_dne\pcc_items_fold1.npy", ) kolom1 = pcc[1]# kolom2 = list(range(1, len(kolom1)+1)
# # print,(kolom
# # print"kolom2", (kolo)
# # # baris.sort(key=lambda x: x[0], reverse=Te)
# # ukuran = pcc.ape
# # print(pcc[550])
# # print(len(p[0]))
# # print("ukuran", ukuran)

In [46]:
def load_data(fold):
    folder_base = '../../Dataset/ml-100k/'
    data_train = f'{folder_base}u{fold}.base'
    data_test = f'{folder_base}u{fold}.test'
    data_global = f'{folder_base}u.data'

    nama_kolom = ['user_id', 'item_id', 'rating', 'timestamp']

    # ambil banyak user dan item dari data global
    read_global = pd.read_csv(data_global, sep='\t', names=nama_kolom)
    u_global = read_global['user_id'].nunique()
    i_global = read_global['item_id'].nunique()

    # load data train dan test
    read_train = pd.read_csv(data_train, sep='\t', names=nama_kolom)
    read_test = pd.read_csv(data_test, sep='\t', names=nama_kolom)

    # matriks rating data global untuk PCC
    mr_global = np.zeros((u_global, i_global))
    # memasukkan data train ke matriks rating global
    for row in read_train.itertuples():
        mr_global[row.user_id-1, row.item_id-1] = row.rating

    # matriks rating data train untuk prior
    user_fold = np.unique(np.concatenate((read_train['user_id'].unique(), read_test['user_id'].unique())))
    item_fold = np.unique(np.concatenate((read_train['item_id'].unique(), read_test['item_id'].unique())))

    # maping user dan item di user_fold dan item_fold ke indeks matriks rating fold
    user_map = {user_id: idx for idx, user_id in enumerate(user_fold)}
    item_map = {item_id: idx for idx, item_id in enumerate(item_fold)}

    b_user_dense = len(user_fold)
    b_item_dense = len(item_fold)

    mr_dense = np.zeros((b_user_dense, b_item_dense))
    for row in read_train.itertuples():
        map_u = user_map[row.user_id]
        map_i = item_map[row.item_id]
        mr_dense[map_u, map_i] = row.rating

    user_target = []
    for row in read_test.itertuples():
        if row.user_id in user_map:
            user_target.append(user_map[row.user_id])
    user_target = sorted(list(set(user_target)))

    # item target untuk prior: gunakan semua item dalam fold
    item_target = list(range(b_item_dense))

    return mr_global, mr_dense, read_test, user_map, item_map, user_target, item_target

In [47]:
# train_ratings_dense.shape
load_data(1)

(array([[5., 3., 4., ..., 0., 0., 0.],
        [4., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [5., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 5., 0., ..., 0., 0., 0.]], shape=(943, 1682)),
 array([[5., 3., 4., ..., 0., 0., 0.],
        [4., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [5., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 5., 0., ..., 0., 0., 0.]], shape=(943, 1682)),
        user_id  item_id  rating  timestamp
 0            1        6       5  887431973
 1            1       10       3  875693118
 2            1       12       5  878542960
 3            1       14       5  874965706
 4            1       17       3  875073198
 ...        ...      ...     ...        ...
 19995      458      648       4  886395899
 19996      458     1101       4  886397931
 19997      459      934       3  879563639
 19998      460       10

In [50]:
import math
import numpy as np
import pandas as pd
import os
import json
from joblib import Parallel, delayed

LIKE_COLUMNS = [
    "user_id",
    "item_id",
    "rating_value",
    "likelihood_user",
    "likelihood_item",
]

POST_COLUMNS = [
    "user_id",
    "item_id",
    "rating_value",
    "posterior_user",
    "posterior_item",
    "score_user",
    "score_item",
    "score_hybrid",
]

PRED_COLUMNS = [
    "user_id",
    "item_id",
    "predicted_rating",
]

TOPN_COLUMNS = [
    "user_id",
    "rank",
    "item_id",
    "predicted_rating",
]


def precompute_fold_assets(fold, alpha, R, output_dir):
    print(f"--- Precompute Fold {fold} ---")
    mr_global, mr_dense, df_test, user_map, item_map, user_target, item_target = load_data(fold)

    pcc_dir = os.path.join(output_dir, "pcc", f"fold_{fold}")
    prior_dir = os.path.join(output_dir, "prior", f"fold_{fold}")

    prior_ub_path = os.path.join(prior_dir, "prior_ub.npy")
    prior_ib_path = os.path.join(prior_dir, "prior_ib.npy")
    user_target_path = os.path.join(prior_dir, "user_target.json")
    item_target_path = os.path.join(prior_dir, "item_target.json")
    pcc_user_path = os.path.join(pcc_dir, "pcc_user.npy")
    pcc_item_path = os.path.join(pcc_dir, "pcc_item.npy")

    if (
        os.path.exists(prior_ub_path)
        and os.path.exists(prior_ib_path)
        and os.path.exists(user_target_path)
        and os.path.exists(item_target_path)
        and os.path.exists(pcc_user_path)
        and os.path.exists(pcc_item_path)
    ):
        return f"Fold {fold} precompute skip (cached)"

    prior_ub, prior_ib = prior_prob(
        mr_dense, target_user=user_target, target_item=item_target, alpha=alpha, R=R
    )

    pcc_user = pcc(mr_global, base="user")
    pcc_item = pcc(mr_global, base="item")

    os.makedirs(pcc_dir, exist_ok=True)
    os.makedirs(prior_dir, exist_ok=True)
    np.save(pcc_user_path, pcc_user)
    np.save(pcc_item_path, pcc_item)
    np.save(prior_ub_path, prior_ub)
    np.save(prior_ib_path, prior_ib)
    with open(user_target_path, "w", encoding="utf-8") as f:
        json.dump(user_target, f)
    with open(item_target_path, "w", encoding="utf-8") as f:
        json.dump(item_target, f)

    return f"Fold {fold} precompute selesai"


def load_fold_assets(fold, output_dir):
    pcc_dir = os.path.join(output_dir, "pcc", f"fold_{fold}")
    prior_dir = os.path.join(output_dir, "prior", f"fold_{fold}")

    prior_ub_path = os.path.join(prior_dir, "prior_ub.npy")
    prior_ib_path = os.path.join(prior_dir, "prior_ib.npy")
    user_target_path = os.path.join(prior_dir, "user_target.json")
    item_target_path = os.path.join(prior_dir, "item_target.json")
    pcc_user_path = os.path.join(pcc_dir, "pcc_user.npy")
    pcc_item_path = os.path.join(pcc_dir, "pcc_item.npy")

    prior_ub = np.load(prior_ub_path)
    prior_ib = np.load(prior_ib_path)
    pcc_user = np.load(pcc_user_path)
    pcc_item = np.load(pcc_item_path)

    with open(user_target_path, "r", encoding="utf-8") as f:
        user_target = json.load(f)
    with open(item_target_path, "r", encoding="utf-8") as f:
        item_target = json.load(f)

    prior_u_map = {item_dense: i for i, item_dense in enumerate(item_target)}
    prior_i_map = {user_dense: i for i, user_dense in enumerate(user_target)}

    return prior_ub, prior_ib, pcc_user, pcc_item, prior_u_map, prior_i_map


def run_likelihood_combo(fold, k_user, k_item, alpha, R, output_dir):
    print(f"--- Likelihood Fold {fold} | k_user={k_user}, k_item={k_item} ---")
    mr_global, mr_dense, df_test, user_map, item_map, user_target, item_target = load_data(fold)
    prior_ub, prior_ib, pcc_user, pcc_item, prior_u_map, prior_i_map = load_fold_assets(
        fold, output_dir
    )

    grouped = list(df_test.groupby("user_id"))
    item_map_inv = {idx: item_id for item_id, idx in item_map.items()}

    like_dir = os.path.join(output_dir, "likelihood", f"fold_{fold}")
    os.makedirs(like_dir, exist_ok=True)

    like_path = os.path.join(
        like_dir, f"likelihood_kuser_{k_user}_kitem_{k_item}.pkl"
    )
    if os.path.exists(like_path):
        return None

    for user_id, group in grouped:
        user_dense = user_map.get(user_id)
        if user_dense is None:
            continue

        unrated_dense = np.where(mr_dense[user_dense, :] == 0)[0]
        user_like_rows = []

        for item_dense in unrated_dense:
            item_id = item_map_inv[item_dense]
            pred, _, detail_y = prediksi_rating(
                mr_global,
                user_id,
                item_id,
                user_map,
                item_map,
                prior_ub,
                prior_ib,
                pcc_user,
                pcc_item,
                prior_u_map=prior_u_map,
                prior_i_map=prior_i_map,
                k_user=k_user,
                k_item=k_item,
                R=R,
            )
            if pred is None:
                continue

            for detail in detail_y:
                user_like_rows.append(
                    {
                        "user_id": user_id,
                        "item_id": item_id,
                        "rating_value": detail["rating_value"],
                        "likelihood_user": detail["likelihood_user"],
                        "likelihood_item": detail["likelihood_item"],
                    }
                )
        if user_like_rows:
            with open(like_path, "ab") as f:
                pickle.dump(pd.DataFrame(user_like_rows), f)

    if not os.path.exists(like_path):
        with open(like_path, "ab") as f:
            pickle.dump(pd.DataFrame(columns=LIKE_COLUMNS), f)
    return None


def run_posterior_combo(fold, k_user, k_item, R, output_dir):
    print(f"--- Posterior Fold {fold} | k_user={k_user}, k_item={k_item} ---")
    _, _, _, user_map, item_map, _, _ = load_data(fold)
    prior_ub, prior_ib, _, _, prior_u_map, prior_i_map = load_fold_assets(fold, output_dir)

    like_dir = os.path.join(output_dir, "likelihood", f"fold_{fold}")
    post_dir = os.path.join(output_dir, "posterior", f"fold_{fold}")
    os.makedirs(post_dir, exist_ok=True)

    like_path = os.path.join(
        like_dir, f"likelihood_kuser_{k_user}_kitem_{k_item}.pkl"
    )
    post_path = os.path.join(
        post_dir, f"posterior_kuser_{k_user}_kitem_{k_item}.pkl"
    )
    if os.path.exists(post_path):
        return None
    if not os.path.exists(like_path):
        print(f"  File tidak ditemukan: {like_path}")
        return None

    data_frames = []
    with open(like_path, "rb") as f:
        while True:
            try:
                data_frames.append(pickle.load(f))
            except EOFError:
                break

    if data_frames:
        df_like = pd.concat(data_frames, ignore_index=True)
    else:
        df_like = pd.DataFrame()

    if df_like.empty:
        with open(post_path, "ab") as f:
            pickle.dump(pd.DataFrame(columns=POST_COLUMNS), f)
        return None
    

    item_dense = df_like["item_id"].map(item_map)
    user_dense = df_like["user_id"].map(user_map)
    idx_prior_user = item_dense.map(prior_u_map)
    idx_prior_item = user_dense.map(prior_i_map)
    valid_mask = idx_prior_user.notna() & idx_prior_item.notna()

    if not valid_mask.any():
        with open(post_path, "ab") as f:
            pickle.dump(pd.DataFrame(columns=POST_COLUMNS), f)
        return None

    df_like = df_like.loc[valid_mask].copy()
    y_index = df_like["rating_value"].to_numpy() - 1
    idx_prior_user = idx_prior_user[valid_mask].astype(int).to_numpy()
    idx_prior_item = idx_prior_item[valid_mask].astype(int).to_numpy()

    prior_u = prior_ub[y_index, idx_prior_user]
    prior_i = prior_ib[y_index, idx_prior_item]
    like_u = df_like["likelihood_user"].to_numpy()
    like_i = df_like["likelihood_item"].to_numpy()

    posterior_u = prior_u * like_u
    posterior_i = prior_i * like_i

    if k_item > 0:
        score_ub = posterior_u ** (1.0 / (k_item + 1))
    else:
        score_ub = np.full_like(posterior_u, -100.0, dtype=float)

    if k_user > 0:
        score_ib = posterior_i ** (1.0 / (k_user + 1))
    else:
        score_ib = np.full_like(posterior_i, -100.0, dtype=float)

    score_hybrid = score_ub * score_ib

    df_post = pd.DataFrame(
        {
            "user_id": df_like["user_id"].to_numpy(),
            "item_id": df_like["item_id"].to_numpy(),
            "rating_value": df_like["rating_value"].to_numpy(),
            "score_user": score_ub,
            "score_item": score_ib,
            "score_hybrid": score_hybrid,
        }
    )
    with open(post_path, "ab") as f:
        pickle.dump(df_post, f)
    return None


def run_prediksi_combo_kr(fold, k_user, k_item, output_dir):
    print(f"--- Prediksi Fold {fold} | k_user={k_user}, k_item={k_item} ---")
    post_dir = os.path.join(output_dir, "posterior", f"fold_{fold}")
    pred_dir = os.path.join(output_dir, "prediksi_hb_kr", f"fold_{fold}")
    os.makedirs(pred_dir, exist_ok=True)

    post_path = os.path.join(
        post_dir, f"posterior_kuser_{k_user}_kitem_{k_item}.pkl"
    )
    pred_path = os.path.join(
        pred_dir, f"pred_kuser_{k_user}_kitem_{k_item}.csv"
    )
    if os.path.exists(pred_path):
        return None
    if not os.path.exists(post_path):
        print(f"  File tidak ditemukan: {post_path}")
        return None

    data_frames = []
    with open(post_path, "rb") as f:
        while True:
            try:
                data_frames.append(pickle.load(f))
            except EOFError:
                break

    if data_frames:
        df_post = pd.concat(data_frames, ignore_index=True)
    else:
        df_post = pd.DataFrame()

    if df_post.empty:
        pd.DataFrame(columns=PRED_COLUMNS).to_csv(pred_path, index=False)
        return None

    rows = []
    for (user_id, item_id), group in df_post.groupby(["user_id", "item_id"]):
        scores = group["score_hybrid"].to_numpy()
        ratings = group["rating_value"].to_numpy()
        max_score = float(np.max(scores))
        idx = np.where(np.isclose(scores, max_score))[0]

        if len(idx) == 1:
            pred = float(ratings[idx[0]])
        else:
            Z = float(np.sum(scores))
            if Z == 0.0:
                pred = float(ratings[idx[0]])
            else:
                pred = float(np.sum(scores * ratings) / Z)

        rows.append(
            {
                "user_id": int(user_id),
                "item_id": int(item_id),
                "predicted_rating": pred,
            }
        )

    pd.DataFrame(rows, columns=PRED_COLUMNS).to_csv(pred_path, index=False)
    return None

def run_prediksi_combo_EV(fold, k_user, k_item, output_dir):
    print(f"--- Prediksi Fold {fold} | k_user={k_user}, k_item={k_item} ---")
    post_dir = os.path.join(output_dir, "posterior", f"fold_{fold}")
    pred_dir = os.path.join(output_dir, "prediksi_hb_EV", f"fold_{fold}")
    os.makedirs(pred_dir, exist_ok=True)

    post_path = os.path.join(
        post_dir, f"posterior_kuser_{k_user}_kitem_{k_item}.pkl"
    )
    pred_path = os.path.join(
        pred_dir, f"pred_kuser_{k_user}_kitem_{k_item}.csv"
    )
    if os.path.exists(pred_path):
        return None
    if not os.path.exists(post_path):
        print(f"  File tidak ditemukan: {post_path}")
        return None

    data_frames = []
    with open(post_path, "rb") as f:
        while True:
            try:
                data_frames.append(pickle.load(f))
            except EOFError:
                break

    if data_frames:
        df_post = pd.concat(data_frames, ignore_index=True)
    else:
        df_post = pd.DataFrame()

    if df_post.empty:
        pd.DataFrame(columns=PRED_COLUMNS).to_csv(pred_path, index=False)
        return None

    rows = []
    for (user_id, item_id), group in df_post.groupby(["user_id", "item_id"]):
        scores = group["score_hybrid"].to_numpy()
        ratings = group["rating_value"].to_numpy()
        max_score = float(np.max(scores))
        idx = np.where(np.isclose(scores, max_score))[0]
        Z = float(np.sum(scores))
        if Z == 0.0:
            pred = float(ratings[idx[0]])
        else:
            pred = float(np.sum(scores * ratings) / Z)

        rows.append(
            {
                "user_id": int(user_id),
                "item_id": int(item_id),
                "predicted_rating": pred,
            }
        )

    pd.DataFrame(rows, columns=PRED_COLUMNS).to_csv(pred_path, index=False)
    return None

def run_prediksi_combo_ubkr(fold, k_user, k_item, output_dir):
    print(f"--- Prediksi Fold {fold} | k_user={k_user}, k_item={k_item} ---")
    post_dir = os.path.join(output_dir, "posterior", f"fold_{fold}")
    pred_dir = os.path.join(output_dir, "prediksi_ub_kr", f"fold_{fold}")
    os.makedirs(pred_dir, exist_ok=True)

    post_path = os.path.join(
        post_dir, f"posterior_kuser_{k_user}_kitem_{k_item}.pkl"
    )
    pred_path = os.path.join(
        pred_dir, f"pred_kitem_{k_item}.csv"
    )
    if os.path.exists(pred_path):
        return None
    if not os.path.exists(post_path):
        print(f"  File tidak ditemukan: {post_path}")
        return None

    data_frames = []
    with open(post_path, "rb") as f:
        while True:
            try:
                data_frames.append(pickle.load(f))
            except EOFError:
                break

    if data_frames:
        df_post = pd.concat(data_frames, ignore_index=True)
    else:
        df_post = pd.DataFrame()

    if df_post.empty:
        pd.DataFrame(columns=PRED_COLUMNS).to_csv(pred_path, index=False)
        return None

    rows = []
    for (user_id, item_id), group in df_post.groupby(["user_id", "item_id"]):
        scores = group["score_user"].to_numpy()
        ratings = group["rating_value"].to_numpy()
        max_score = float(np.max(scores))
        idx = np.where(np.isclose(scores, max_score))[0]

        if len(idx) == 1:
            pred = float(ratings[idx[0]])
        else:
            Z = float(np.sum(scores))
            if Z == 0.0:
                pred = float(ratings[idx[0]])
            else:
                pred = float(np.sum(scores * ratings) / Z)

        rows.append(
            {
                "user_id": int(user_id),
                "item_id": int(item_id),
                "predicted_rating": pred,
            }
        )

    pd.DataFrame(rows, columns=PRED_COLUMNS).to_csv(pred_path, index=False)
    return None

def run_prediksi_combo_ubEV(fold, k_user, k_item, output_dir):
    print(f"--- Prediksi Fold {fold} | k_user={k_user}, k_item={k_item} ---")
    post_dir = os.path.join(output_dir, "posterior", f"fold_{fold}")
    pred_dir = os.path.join(output_dir, "prediksi_ub_EV", f"fold_{fold}")
    os.makedirs(pred_dir, exist_ok=True)

    post_path = os.path.join(
        post_dir, f"posterior_kuser_{k_user}_kitem_{k_item}.pkl"
    )
    pred_path = os.path.join(
        pred_dir, f"pred_kitem_{k_item}.csv"
    )
    if os.path.exists(pred_path):
        return None
    if not os.path.exists(post_path):
        print(f"  File tidak ditemukan: {post_path}")
        return None

    data_frames = []
    with open(post_path, "rb") as f:
        while True:
            try:
                data_frames.append(pickle.load(f))
            except EOFError:
                break

    if data_frames:
        df_post = pd.concat(data_frames, ignore_index=True)
    else:
        df_post = pd.DataFrame()

    if df_post.empty:
        pd.DataFrame(columns=PRED_COLUMNS).to_csv(pred_path, index=False)
        return None

    rows = []
    for (user_id, item_id), group in df_post.groupby(["user_id", "item_id"]):
        scores = group["score_user"].to_numpy()
        ratings = group["rating_value"].to_numpy()
        max_score = float(np.max(scores))
        idx = np.where(np.isclose(scores, max_score))[0]
        Z = float(np.sum(scores))
        if Z == 0.0:
            pred = float(ratings[idx[0]])
        else:
            pred = float(np.sum(scores * ratings) / Z)

        rows.append(
            {
                "user_id": int(user_id),
                "item_id": int(item_id),
                "predicted_rating": pred,
            }
        )

    pd.DataFrame(rows, columns=PRED_COLUMNS).to_csv(pred_path, index=False)
    return None

def run_prediksi_combo_ibkr(fold, k_user, k_item, output_dir):
    print(f"--- Prediksi Fold {fold} | k_user={k_user}, k_item={k_item} ---")
    post_dir = os.path.join(output_dir, "posterior", f"fold_{fold}")
    pred_dir = os.path.join(output_dir, "prediksi_ib_kr", f"fold_{fold}")
    os.makedirs(pred_dir, exist_ok=True)

    post_path = os.path.join(
        post_dir, f"posterior_kuser_{k_user}_kitem_{k_item}.pkl"
    )
    pred_path = os.path.join(
        pred_dir, f"pred_kuser_{k_user}.csv"
    )
    if os.path.exists(pred_path):
        return None
    if not os.path.exists(post_path):
        print(f"  File tidak ditemukan: {post_path}")
        return None

    data_frames = []
    with open(post_path, "rb") as f:
        while True:
            try:
                data_frames.append(pickle.load(f))
            except EOFError:
                break

    if data_frames:
        df_post = pd.concat(data_frames, ignore_index=True)
    else:
        df_post = pd.DataFrame()

    if df_post.empty:
        pd.DataFrame(columns=PRED_COLUMNS).to_csv(pred_path, index=False)
        return None

    rows = []
    for (user_id, item_id), group in df_post.groupby(["user_id", "item_id"]):
        scores = group["score_item"].to_numpy()
        ratings = group["rating_value"].to_numpy()
        max_score = float(np.max(scores))
        idx = np.where(np.isclose(scores, max_score))[0]

        if len(idx) == 1:
            pred = float(ratings[idx[0]])
        else:
            Z = float(np.sum(scores))
            if Z == 0.0:
                pred = float(ratings[idx[0]])
            else:
                pred = float(np.sum(scores * ratings) / Z)

        rows.append(
            {
                "user_id": int(user_id),
                "item_id": int(item_id),
                "predicted_rating": pred,
            }
        )

    pd.DataFrame(rows, columns=PRED_COLUMNS).to_csv(pred_path, index=False)
    return None

def run_prediksi_combo_ibEV(fold, k_user, k_item, output_dir):
    print(f"--- Prediksi Fold {fold} | k_user={k_user}, k_item={k_item} ---")
    post_dir = os.path.join(output_dir, "posterior", f"fold_{fold}")
    pred_dir = os.path.join(output_dir, "prediksi_ib_EV", f"fold_{fold}")
    os.makedirs(pred_dir, exist_ok=True)

    post_path = os.path.join(
        post_dir, f"posterior_kuser_{k_user}_kitem_{k_item}.pkl"
    )
    pred_path = os.path.join(
        pred_dir, f"pred_kuser_{k_user}.csv"
    )
    if os.path.exists(pred_path):
        return None
    if not os.path.exists(post_path):
        print(f"  File tidak ditemukan: {post_path}")
        return None

    data_frames = []
    with open(post_path, "rb") as f:
        while True:
            try:
                data_frames.append(pickle.load(f))
            except EOFError:
                break

    if data_frames:
        df_post = pd.concat(data_frames, ignore_index=True)
    else:
        df_post = pd.DataFrame()

    if df_post.empty:
        pd.DataFrame(columns=PRED_COLUMNS).to_csv(pred_path, index=False)
        return None

    rows = []
    for (user_id, item_id), group in df_post.groupby(["user_id", "item_id"]):
        scores = group["score_item"].to_numpy()
        ratings = group["rating_value"].to_numpy()
        max_score = float(np.max(scores))
        idx = np.where(np.isclose(scores, max_score))[0]
        Z = float(np.sum(scores))
        if Z == 0.0:
            pred = float(ratings[idx[0]])
        else:
            pred = float(np.sum(scores * ratings) / Z)

        rows.append(
            {
                "user_id": int(user_id),
                "item_id": int(item_id),
                "predicted_rating": pred,
            }
        )

    pd.DataFrame(rows, columns=PRED_COLUMNS).to_csv(pred_path, index=False)
    return None


def run_topn_combo_kr(fold, k_user, k_item, top_n, output_dir):
    print(f"--- TopN Fold {fold} | k_user={k_user}, k_item={k_item} ---")
    pred_dir = os.path.join(output_dir, "prediksi_hb_kr", f"fold_{fold}")
    topn_dir = os.path.join(output_dir, "topn_hb_kr", f"fold_{fold}")
    os.makedirs(topn_dir, exist_ok=True)

    pred_path = os.path.join(
        pred_dir, f"pred_kuser_{k_user}_kitem_{k_item}.csv"
    )
    topn_path = os.path.join(
        topn_dir, f"topn_{top_n}_kuser_{k_user}_kitem_{k_item}.csv"
    )
    if os.path.exists(topn_path):
        return None
    if not os.path.exists(pred_path):
        print(f"  File tidak ditemukan: {pred_path}")
        return None

    df_pred = pd.read_csv(pred_path)
    if df_pred.empty:
        pd.DataFrame(columns=TOPN_COLUMNS).to_csv(topn_path, index=False)
        return None

    rows = []
    for user_id, group in df_pred.groupby("user_id"):
        group_sorted = group.sort_values(
            ["predicted_rating", "item_id"], ascending=[False, True]
        )
        for rank, (_, row) in enumerate(group_sorted.head(top_n).iterrows(), start=1):
            rows.append(
                {
                    "user_id": int(row["user_id"]),
                    "rank": int(rank),
                    "item_id": int(row["item_id"]),
                    "predicted_rating": float(row["predicted_rating"]),
                }
            )

    pd.DataFrame(rows).to_csv(topn_path, index=False)
    return None

def run_topn_combo_EV(fold, k_user, k_item, top_n, output_dir):
    print(f"--- TopN Fold {fold} | k_user={k_user}, k_item={k_item} ---")
    pred_dir = os.path.join(output_dir, "prediksi_hb_EV", f"fold_{fold}")
    topn_dir = os.path.join(output_dir, "topn_hb_EV", f"fold_{fold}")
    os.makedirs(topn_dir, exist_ok=True)

    pred_path = os.path.join(
        pred_dir, f"pred_kuser_{k_user}_kitem_{k_item}.csv"
    )
    topn_path = os.path.join(
        topn_dir, f"topn_{top_n}_kuser_{k_user}_kitem_{k_item}.csv"
    )
    if os.path.exists(topn_path):
        return None
    if not os.path.exists(pred_path):
        print(f"  File tidak ditemukan: {pred_path}")
        return None

    df_pred = pd.read_csv(pred_path)
    if df_pred.empty:
        pd.DataFrame(columns=TOPN_COLUMNS).to_csv(topn_path, index=False)
        return None

    rows = []
    for user_id, group in df_pred.groupby("user_id"):
        group_sorted = group.sort_values(
            ["predicted_rating", "item_id"], ascending=[False, True]
        )
        for rank, (_, row) in enumerate(group_sorted.head(top_n).iterrows(), start=1):
            rows.append(
                {
                    "user_id": int(row["user_id"]),
                    "rank": int(rank),
                    "item_id": int(row["item_id"]),
                    "predicted_rating": float(row["predicted_rating"]),
                }
            )

    pd.DataFrame(rows).to_csv(topn_path, index=False)
    return None

def run_topn_combo_ubkr(fold, k_item, top_n, output_dir):
    print(f"--- TopN Fold {fold} | k_item={k_item} ---")
    pred_dir = os.path.join(output_dir, "prediksi_ub_kr", f"fold_{fold}")
    topn_dir = os.path.join(output_dir, "topn_ub_kr", f"fold_{fold}")
    os.makedirs(topn_dir, exist_ok=True)

    pred_path = os.path.join(
        pred_dir, f"pred_kitem_{k_item}.csv"
    )
    topn_path = os.path.join(
        topn_dir, f"topn_{top_n}_kitem_{k_item}.csv"
    )
    if os.path.exists(topn_path):
        return None
    if not os.path.exists(pred_path):
        print(f"  File tidak ditemukan: {pred_path}")
        return None

    df_pred = pd.read_csv(pred_path)
    if df_pred.empty:
        pd.DataFrame(columns=TOPN_COLUMNS).to_csv(topn_path, index=False)
        return None

    rows = []
    for user_id, group in df_pred.groupby("user_id"):
        group_sorted = group.sort_values(
            ["predicted_rating", "item_id"], ascending=[False, True]
        )
        for rank, (_, row) in enumerate(group_sorted.head(top_n).iterrows(), start=1):
            rows.append(
                {
                    "user_id": int(row["user_id"]),
                    "rank": int(rank),
                    "item_id": int(row["item_id"]),
                    "predicted_rating": float(row["predicted_rating"]),
                }
            )

    pd.DataFrame(rows).to_csv(topn_path, index=False)
    return None

def run_topn_combo_ubEV(fold, k_item, top_n, output_dir):
    print(f"--- TopN Fold {fold} | k_item={k_item} ---")
    pred_dir = os.path.join(output_dir, "prediksi_ub_EV", f"fold_{fold}")
    topn_dir = os.path.join(output_dir, "topn_ub_EV", f"fold_{fold}")
    os.makedirs(topn_dir, exist_ok=True)

    pred_path = os.path.join(
        pred_dir, f"pred_kitem_{k_item}.csv"
    )
    topn_path = os.path.join(
        topn_dir, f"topn_{top_n}_kitem_{k_item}.csv"
    )
    if os.path.exists(topn_path):
        return None
    if not os.path.exists(pred_path):
        print(f"  File tidak ditemukan: {pred_path}")
        return None

    df_pred = pd.read_csv(pred_path)
    if df_pred.empty:
        pd.DataFrame(columns=TOPN_COLUMNS).to_csv(topn_path, index=False)
        return None

    rows = []
    for user_id, group in df_pred.groupby("user_id"):
        group_sorted = group.sort_values(
            ["predicted_rating", "item_id"], ascending=[False, True]
        )
        for rank, (_, row) in enumerate(group_sorted.head(top_n).iterrows(), start=1):
            rows.append(
                {
                    "user_id": int(row["user_id"]),
                    "rank": int(rank),
                    "item_id": int(row["item_id"]),
                    "predicted_rating": float(row["predicted_rating"]),
                }
            )

    pd.DataFrame(rows).to_csv(topn_path, index=False)
    return None

def run_topn_combo_ibkr(fold, k_user, top_n, output_dir):
    print(f"--- TopN Fold {fold} | k_user={k_user} ---")
    pred_dir = os.path.join(output_dir, "prediksi_ib_kr", f"fold_{fold}")
    topn_dir = os.path.join(output_dir, "topn_ib_kr", f"fold_{fold}")
    os.makedirs(topn_dir, exist_ok=True)

    pred_path = os.path.join(
        pred_dir, f"pred_kuser_{k_user}.csv"
    )
    topn_path = os.path.join(
        topn_dir, f"topn_{top_n}_kuser_{k_user}.csv"
    )
    if os.path.exists(topn_path):
        return None
    if not os.path.exists(pred_path):
        print(f"  File tidak ditemukan: {pred_path}")
        return None

    df_pred = pd.read_csv(pred_path)
    if df_pred.empty:
        pd.DataFrame(columns=TOPN_COLUMNS).to_csv(topn_path, index=False)
        return None

    rows = []
    for user_id, group in df_pred.groupby("user_id"):
        group_sorted = group.sort_values(
            ["predicted_rating", "item_id"], ascending=[False, True]
        )
        for rank, (_, row) in enumerate(group_sorted.head(top_n).iterrows(), start=1):
            rows.append(
                {
                    "user_id": int(row["user_id"]),
                    "rank": int(rank),
                    "item_id": int(row["item_id"]),
                    "predicted_rating": float(row["predicted_rating"]),
                }
            )

    pd.DataFrame(rows).to_csv(topn_path, index=False)
    return None

def run_topn_combo_ibEV(fold, k_user, top_n, output_dir):
    print(f"--- TopN Fold {fold} | k_user={k_user} ---")
    pred_dir = os.path.join(output_dir, "prediksi_ib_EV", f"fold_{fold}")
    topn_dir = os.path.join(output_dir, "topn_ib_EV", f"fold_{fold}")
    os.makedirs(topn_dir, exist_ok=True)

    pred_path = os.path.join(
        pred_dir, f"pred_kuser_{k_user}.csv"
    )
    topn_path = os.path.join(
        topn_dir, f"topn_{top_n}_kuser_{k_user}.csv"
    )
    if os.path.exists(topn_path):
        return None
    if not os.path.exists(pred_path):
        print(f"  File tidak ditemukan: {pred_path}")
        return None

    df_pred = pd.read_csv(pred_path)
    if df_pred.empty:
        pd.DataFrame(columns=TOPN_COLUMNS).to_csv(topn_path, index=False)
        return None

    rows = []
    for user_id, group in df_pred.groupby("user_id"):
        group_sorted = group.sort_values(
            ["predicted_rating", "item_id"], ascending=[False, True]
        )
        for rank, (_, row) in enumerate(group_sorted.head(top_n).iterrows(), start=1):
            rows.append(
                {
                    "user_id": int(row["user_id"]),
                    "rank": int(rank),
                    "item_id": int(row["item_id"]),
                    "predicted_rating": float(row["predicted_rating"]),
                }
            )

    pd.DataFrame(rows).to_csv(topn_path, index=False)
    return None

def run_cv_5fold_grid(
    k_values=None,
    k_folds=5,
    alpha=0.01,
    R=5,
    top_n=100,
    output_dir="hasil_eval_ndcg",
    n_jobs=8,
    precompute_jobs=None,
):
    if k_values is None:
        k_values = [10, 20, 30, 40, 50, 100, 150, 200]

    if precompute_jobs is None:
        precompute_jobs = n_jobs

    print(
        f"Inisialisasi uji {k_folds}-fold CV (staged, n_jobs={n_jobs}, precompute_jobs={precompute_jobs})..."
    )
    os.makedirs(output_dir, exist_ok=True)

    print("Tahap A: precompute prior dan ITR per fold...")
    Parallel(n_jobs=precompute_jobs, verbose=10)(
        delayed(precompute_fold_assets)(fold, alpha, R, output_dir)
        for fold in range(1, k_folds + 1)
    )

    print("Tahap B1: Likelihood parallel untuk semua kombinasi...")
    Parallel(n_jobs=n_jobs, verbose=10)(
        delayed(run_likelihood_combo)(fold, k_user, k_item, alpha, R, output_dir)
        for fold in range(1, k_folds + 1)
        for k_user in k_values
        for k_item in k_values
    )

    print("Tahap B2: Posterior parallel untuk semua kombinasi...")
    Parallel(n_jobs=n_jobs, verbose=10)(
        delayed(run_posterior_combo)(fold, k_user, k_item, R, output_dir)
        for fold in range(1, k_folds + 1)
        for k_user in k_values
        for k_item in k_values
    )

    print("Tahap B3A: Prediksi parallel untuk semua kombinasi...")
    Parallel(n_jobs=10, verbose=10)(
        delayed(run_prediksi_combo_kr)(fold, k_user, k_item, output_dir)
        for fold in range(1, k_folds + 1)
        for k_user in k_values
        for k_item in k_values
    )

    print("Tahap B3B: Prediksi parallel untuk semua kombinasi...")
    Parallel(n_jobs=10, verbose=10)(
        delayed(run_prediksi_combo_EV)(fold, k_user, k_item, output_dir)
        for fold in range(1, k_folds + 1)
        for k_user in k_values
        for k_item in k_values
    )

    print("Tahap B3C: Prediksi parallel untuk semua kombinasi...")
    Parallel(n_jobs=10, verbose=10)(
        delayed(run_prediksi_combo_ubkr)(fold, k_user, k_item, output_dir)
        for fold in range(1, k_folds + 1)
        for k_user in k_values
        for k_item in k_values
    )

    print("Tahap B3D: Prediksi parallel untuk semua kombinasi...")
    Parallel(n_jobs=10, verbose=10)(
        delayed(run_prediksi_combo_ubEV)(fold, k_user, k_item, output_dir)
        for fold in range(1, k_folds + 1)
        for k_user in k_values
        for k_item in k_values
    )

    print("Tahap B3E: Prediksi parallel untuk semua kombinasi...")
    Parallel(n_jobs=10, verbose=10)(
        delayed(run_prediksi_combo_ibkr)(fold, k_user, k_item, output_dir)
        for fold in range(1, k_folds + 1)
        for k_user in k_values
        for k_item in k_values
    )

    print("Tahap B3F: Prediksi parallel untuk semua kombinasi...")
    Parallel(n_jobs=10, verbose=10)(
        delayed(run_prediksi_combo_ibEV)(fold, k_user, k_item, output_dir)
        for fold in range(1, k_folds + 1)
        for k_user in k_values
        for k_item in k_values
    )

    print("Tahap B4A: TopN parallel untuk semua kombinasi...")
    Parallel(n_jobs=n_jobs, verbose=10)(
        delayed(run_topn_combo_kr)(fold, k_user, k_item, top_n, output_dir)
        for fold in range(1, k_folds + 1)
        for k_user in k_values
        for k_item in k_values
    )
    print("Tahap B4B: TopN parallel untuk semua kombinasi...")
    Parallel(n_jobs=n_jobs, verbose=10)(
        delayed(run_topn_combo_EV)(fold, k_user, k_item, top_n, output_dir)
        for fold in range(1, k_folds + 1)
        for k_user in k_values
        for k_item in k_values
    )

    print("Tahap B4C: TopN parallel untuk semua kombinasi...")
    Parallel(n_jobs=n_jobs, verbose=10)(
        delayed(run_topn_combo_ubkr)(fold, k_item, top_n, output_dir)
        for fold in range(1, k_folds + 1)
        for k_user in k_values
        for k_item in k_values
    )
    print("Tahap B4D: TopN parallel untuk semua kombinasi...")
    Parallel(n_jobs=n_jobs, verbose=10)(
        delayed(run_topn_combo_ubEV)(fold, k_item, top_n, output_dir)
        for fold in range(1, k_folds + 1)
        for k_user in k_values
        for k_item in k_values
    )
    print("Tahap B4E: TopN parallel untuk semua kombinasi...")
    Parallel(n_jobs=n_jobs, verbose=10)(
        delayed(run_topn_combo_ibkr)(fold, k_user, top_n, output_dir)
        for fold in range(1, k_folds + 1)
        for k_user in k_values
        for k_item in k_values
    )
    print("Tahap B4F: TopN parallel untuk semua kombinasi...")
    Parallel(n_jobs=n_jobs, verbose=10)(
        delayed(run_topn_combo_ibEV)(fold, k_user, top_n, output_dir)
        for fold in range(1, k_folds + 1)
        for k_user in k_values
        for k_item in k_values
    )

    print("Selesai CV staged.")



def eval_ndcg_from_saved_hbkr(
    k_values=None, k_folds=5, k=20, output_dir="hasil_eval_ndcg"
):
    if k_values is None:
        k_values = [10, 20, 30, 40, 50, 100, 150, 200]

    # Menentukan direktori penyimpanan di dalam subfolder 'ndcg_hybrid_kr'
    save_dir = os.path.join(output_dir, "ndcg_hybrid_kr")
    per_user_dir = os.path.join(save_dir, "per_user_ndcg", f"fold_{k_folds}")
    os.makedirs(per_user_dir, exist_ok=True)

    print("Evaluasi NDCG dari hasil top-N yang tersimpan...")
    results = []

    k_max = int(k)
    if k_max < 1:
        print("k harus >= 1")
        return results

    ground_truth_per_fold = {}
    for fold in range(1, k_folds + 1):
        _, _, df_test, _, _, _, _ = load_data(fold)
        ground_truth_per_fold[fold] = {
            user_id: set(group["item_id"].to_numpy())
            for user_id, group in df_test.groupby("user_id")
        }

    for k_user in k_values:
        for k_item in k_values:
            fold_scores_per_k = {k_eval: [] for k_eval in range(1, k_max + 1)}
            all_user_ndcgs = []  # Menampung seluruh data per_user untuk pasangan k_user & k_item

            for fold in range(1, k_folds + 1):
                # Tetap membaca top-N yang tersimpan dari folder topn_kr
                topn_path = os.path.join(
                    output_dir,
                    "topn_hb_kr",
                    f"fold_{fold}",
                    f"topn_{k_max}_kuser_{k_user}_kitem_{k_item}.csv",
                )
                if not os.path.exists(topn_path):
                    print(f"  File tidak ditemukan: {topn_path}")
                    for k_eval in range(1, k_max + 1):
                        fold_scores_per_k[k_eval].append(0.0)
                    continue

                df_topn = pd.read_csv(topn_path)
                if df_topn.empty:
                    for k_eval in range(1, k_max + 1):
                        fold_scores_per_k[k_eval].append(0.0)
                    continue

                gt_map = ground_truth_per_fold[fold]
                user_ranked = {
                    user_id: group.sort_values("rank")["item_id"].to_numpy()
                    for user_id, group in df_topn.groupby("user_id")
                }

                for k_eval in range(1, k_max + 1):
                    user_ndcgs = []
                    for user_id, ranked_items in user_ranked.items():
                        gt_items = gt_map.get(user_id)
                        if not gt_items:
                            continue

                        dcg = 0.0
                        for rank, item_id in enumerate(
                            ranked_items[:k_eval], start=1
                        ):
                            dcg += (1 / math.log2(rank + 1)) * (
                                1 if item_id in gt_items else 0
                            )

                        idcg_len = min(k_eval, len(gt_items))
                        if idcg_len == 0:
                            continue

                        idcg = 0.0
                        for rank in range(1, idcg_len + 1):
                            idcg += 1 / math.log2(rank + 1)

                        ndcg = dcg / idcg if idcg > 0 else 0.0
                        user_ndcgs.append(
                            {
                                "user_id": user_id,
                                "fold": fold,
                                "k_user": k_user,
                                "k_item": k_item,
                                "k": k_eval,
                                "ndcg": ndcg,
                            }
                        )

                    all_user_ndcgs.extend(user_ndcgs)

                    fold_ndcg = (
                        float(np.mean([x["ndcg"] for x in user_ndcgs]))
                        if user_ndcgs
                        else 0.0
                    )
                    fold_scores_per_k[k_eval].append(fold_ndcg)

            # Simpan detail per user ke file CSV di dalam subfolder: ndcg_hybrid_kr/per_user_ndcg/fold_X/
            if all_user_ndcgs:
                per_user_path = os.path.join(
                    per_user_dir, f"k_user_{k_user}_k_item_{k_item}.csv"
                )
                pd.DataFrame(all_user_ndcgs).to_csv(
                    per_user_path, index=False
                )

            print(f"  k_user={k_user}, k_item={k_item}, topn={k_max}")

            for k_eval in range(1, k_max + 1):
                fold_scores = fold_scores_per_k[k_eval]
                mean_ndcg = (
                    float(np.mean(fold_scores)) if fold_scores else 0.0
                )
                results.append(
                    {
                        "k_user": k_user,
                        "k_item": k_item,
                        "k": k_eval,
                        "mean_ndcg": mean_ndcg,
                        "fold_scores": fold_scores,
                    }
                )

    # Simpan ringkasan hasil evaluasi keseluruhan ke CSV di dalam subfolder: ndcg_hybrid_kr/summary_ndcg.csv
    summary_path = os.path.join(save_dir, "summary_ndcg.csv")
    pd.DataFrame(results).to_csv(summary_path, index=False)

    print(f"Detail per user disimpan di folder: {per_user_dir}")
    print(f"Ringkasan evaluasi disimpan di: {summary_path}")

    return results

def eval_ndcg_from_saved_hbev(
    k_values=None, k_folds=5, k=20, output_dir="hasil_eval_ndcg"
):
    if k_values is None:
        k_values = [10, 20, 30, 40, 50, 100, 150, 200]

    # Menentukan direktori penyimpanan di dalam subfolder 'ndcg_hybrid_EV'
    save_dir = os.path.join(output_dir, "ndcg_hybrid_EV")
    per_user_dir = os.path.join(save_dir, "per_user_ndcg", f"fold_{k_folds}")
    os.makedirs(per_user_dir, exist_ok=True)

    print("Evaluasi NDCG dari hasil top-N yang tersimpan...")
    results = []

    k_max = int(k)
    if k_max < 1:
        print("k harus >= 1")
        return results

    ground_truth_per_fold = {}
    for fold in range(1, k_folds + 1):
        _, _, df_test, _, _, _, _ = load_data(fold)
        ground_truth_per_fold[fold] = {
            user_id: set(group["item_id"].to_numpy())
            for user_id, group in df_test.groupby("user_id")
        }

    for k_user in k_values:
        for k_item in k_values:
            fold_scores_per_k = {k_eval: [] for k_eval in range(1, k_max + 1)}
            all_user_ndcgs = []  # Menampung seluruh data per_user untuk pasangan k_user & k_item

            for fold in range(1, k_folds + 1):
                # Tetap membaca top-N yang tersimpan dari folder topn_kr
                topn_path = os.path.join(
                    output_dir,
                    "topn_hb_EV",
                    f"fold_{fold}",
                    f"topn_{k_max}_kuser_{k_user}_kitem_{k_item}.csv",
                )
                if not os.path.exists(topn_path):
                    print(f"  File tidak ditemukan: {topn_path}")
                    for k_eval in range(1, k_max + 1):
                        fold_scores_per_k[k_eval].append(0.0)
                    continue

                df_topn = pd.read_csv(topn_path)
                if df_topn.empty:
                    for k_eval in range(1, k_max + 1):
                        fold_scores_per_k[k_eval].append(0.0)
                    continue

                gt_map = ground_truth_per_fold[fold]
                user_ranked = {
                    user_id: group.sort_values("rank")["item_id"].to_numpy()
                    for user_id, group in df_topn.groupby("user_id")
                }

                for k_eval in range(1, k_max + 1):
                    user_ndcgs = []
                    for user_id, ranked_items in user_ranked.items():
                        gt_items = gt_map.get(user_id)
                        if not gt_items:
                            continue

                        dcg = 0.0
                        for rank, item_id in enumerate(
                            ranked_items[:k_eval], start=1
                        ):
                            dcg += (1 / math.log2(rank + 1)) * (
                                1 if item_id in gt_items else 0
                            )

                        idcg_len = min(k_eval, len(gt_items))
                        if idcg_len == 0:
                            continue

                        idcg = 0.0
                        for rank in range(1, idcg_len + 1):
                            idcg += 1 / math.log2(rank + 1)

                        ndcg = dcg / idcg if idcg > 0 else 0.0
                        user_ndcgs.append(
                            {
                                "user_id": user_id,
                                "fold": fold,
                                "k_user": k_user,
                                "k_item": k_item,
                                "k": k_eval,
                                "ndcg": ndcg,
                            }
                        )

                    all_user_ndcgs.extend(user_ndcgs)

                    fold_ndcg = (
                        float(np.mean([x["ndcg"] for x in user_ndcgs]))
                        if user_ndcgs
                        else 0.0
                    )
                    fold_scores_per_k[k_eval].append(fold_ndcg)

            # Simpan detail per user ke file CSV di dalam subfolder: ndcg_hybrid_kr/per_user_ndcg/fold_X/
            if all_user_ndcgs:
                per_user_path = os.path.join(
                    per_user_dir, f"k_user_{k_user}_k_item_{k_item}.csv"
                )
                pd.DataFrame(all_user_ndcgs).to_csv(
                    per_user_path, index=False
                )

            print(f"  k_user={k_user}, k_item={k_item}, topn={k_max}")

            for k_eval in range(1, k_max + 1):
                fold_scores = fold_scores_per_k[k_eval]
                mean_ndcg = (
                    float(np.mean(fold_scores)) if fold_scores else 0.0
                )
                results.append(
                    {
                        "k_user": k_user,
                        "k_item": k_item,
                        "k": k_eval,
                        "mean_ndcg": mean_ndcg,
                        "fold_scores": fold_scores,
                    }
                )

    # Simpan ringkasan hasil evaluasi keseluruhan ke CSV di dalam subfolder: ndcg_hybrid_kr/summary_ndcg.csv
    summary_path = os.path.join(save_dir, "summary_ndcg.csv")
    pd.DataFrame(results).to_csv(summary_path, index=False)

    print(f"Detail per user disimpan di folder: {per_user_dir}")
    print(f"Ringkasan evaluasi disimpan di: {summary_path}")

    return 

def eval_ndcg_from_saved_ubkr(
    k_values=None, k_folds=5, k=20, output_dir="hasil_eval_ndcg"
):
    if k_values is None:
        k_values = [10, 20, 30, 40, 50, 100, 150, 200]

    # Menentukan direktori penyimpanan di dalam subfolder 'ndcg_ub_kr'
    save_dir = os.path.join(output_dir, "ndcg_ub_kr")
    per_user_dir = os.path.join(save_dir, "per_user_ndcg", f"fold_{k_folds}")
    os.makedirs(per_user_dir, exist_ok=True)

    print("Evaluasi NDCG dari hasil top-N yang tersimpan...")
    results = []

    k_max = int(k)
    if k_max < 1:
        print("k harus >= 1")
        return results

    ground_truth_per_fold = {}
    for fold in range(1, k_folds + 1):
        _, _, df_test, _, _, _, _ = load_data(fold)
        ground_truth_per_fold[fold] = {
            user_id: set(group["item_id"].to_numpy())
            for user_id, group in df_test.groupby("user_id")
        }

    for k_item in k_values:
        fold_scores_per_k = {k_eval: [] for k_eval in range(1, k_max + 1)}
        all_user_ndcgs = []  # Menampung seluruh data per_user untuk pasangan k_user & k_item

        for fold in range(1, k_folds + 1):
            # Tetap membaca top-N yang tersimpan dari folder topn_kr
            topn_path = os.path.join(
                output_dir,
                "topn_ub_kr",
                f"fold_{fold}",
                f"topn_{k_max}_kitem_{k_item}.csv",
            )
            if not os.path.exists(topn_path):
                print(f"  File tidak ditemukan: {topn_path}")
                for k_eval in range(1, k_max + 1):
                    fold_scores_per_k[k_eval].append(0.0)
                continue

            df_topn = pd.read_csv(topn_path)
            if df_topn.empty:
                for k_eval in range(1, k_max + 1):
                    fold_scores_per_k[k_eval].append(0.0)
                continue

            gt_map = ground_truth_per_fold[fold]
            user_ranked = {
                user_id: group.sort_values("rank")["item_id"].to_numpy()
                for user_id, group in df_topn.groupby("user_id")
            }

            for k_eval in range(1, k_max + 1):
                user_ndcgs = []
                for user_id, ranked_items in user_ranked.items():
                    gt_items = gt_map.get(user_id)
                    if not gt_items:
                        continue

                    dcg = 0.0
                    for rank, item_id in enumerate(
                        ranked_items[:k_eval], start=1
                    ):
                        dcg += (1 / math.log2(rank + 1)) * (
                            1 if item_id in gt_items else 0
                        )

                    idcg_len = min(k_eval, len(gt_items))
                    if idcg_len == 0:
                        continue

                    idcg = 0.0
                    for rank in range(1, idcg_len + 1):
                        idcg += 1 / math.log2(rank + 1)

                    ndcg = dcg / idcg if idcg > 0 else 0.0
                    user_ndcgs.append(
                        {
                            "user_id": user_id,
                            "fold": fold,
                            "k_item": k_item,
                            "k": k_eval,
                            "ndcg": ndcg,
                        }
                    )

                all_user_ndcgs.extend(user_ndcgs)

                fold_ndcg = (
                    float(np.mean([x["ndcg"] for x in user_ndcgs]))
                    if user_ndcgs
                    else 0.0
                )
                fold_scores_per_k[k_eval].append(fold_ndcg)

        # Simpan detail per user ke file CSV di dalam subfolder: ndcg_ub_kr/per_user_ndcg/fold_X/
        if all_user_ndcgs:
            per_user_path = os.path.join(
                per_user_dir, f"k_item_{k_item}.csv"
            )
            pd.DataFrame(all_user_ndcgs).to_csv(
                per_user_path, index=False
            )

        print(f"  k_item={k_item}, topn={k_max}")

        for k_eval in range(1, k_max + 1):
            fold_scores = fold_scores_per_k[k_eval]
            mean_ndcg = (
                float(np.mean(fold_scores)) if fold_scores else 0.0
            )
            results.append(
                {
                    "k_item": k_item,
                    "k": k_eval,
                    "mean_ndcg": mean_ndcg,
                    "fold_scores": fold_scores,
                }
            )

    # Simpan ringkasan hasil evaluasi keseluruhan ke CSV di dalam subfolder: ndcg_hybrid_kr/summary_ndcg.csv
    summary_path = os.path.join(save_dir, "summary_ndcg.csv")
    pd.DataFrame(results).to_csv(summary_path, index=False)

    print(f"Detail per user disimpan di folder: {per_user_dir}")
    print(f"Ringkasan evaluasi disimpan di: {summary_path}")

    return results

def eval_ndcg_from_saved_ibkr(
    k_values=None, k_folds=5, k=20, output_dir="hasil_eval_ndcg"
):
    if k_values is None:
        k_values = [10, 20, 30, 40, 50, 100, 150, 200]

    # Menentukan direktori penyimpanan di dalam subfolder 'ndcg_hybrid_kr'
    save_dir = os.path.join(output_dir, "ndcg_ib_kr")
    per_user_dir = os.path.join(save_dir, "per_user_ndcg", f"fold_{k_folds}")
    os.makedirs(per_user_dir, exist_ok=True)

    print("Evaluasi NDCG dari hasil top-N yang tersimpan...")
    results = []

    k_max = int(k)
    if k_max < 1:
        print("k harus >= 1")
        return results

    ground_truth_per_fold = {}
    for fold in range(1, k_folds + 1):
        _, _, df_test, _, _, _, _ = load_data(fold)
        ground_truth_per_fold[fold] = {
            user_id: set(group["item_id"].to_numpy())
            for user_id, group in df_test.groupby("user_id")
        }

    for k_user in k_values:
        fold_scores_per_k = {k_eval: [] for k_eval in range(1, k_max + 1)}
        all_user_ndcgs = []  # Menampung seluruh data per_user untuk pasangan k_user & k_item

        for fold in range(1, k_folds + 1):
            # Tetap membaca top-N yang tersimpan dari folder topn_kr
            topn_path = os.path.join(
                output_dir,
                "topn_ib_kr",
                f"fold_{fold}",
                f"topn_{k_max}_kuser_{k_user}.csv",
            )
            if not os.path.exists(topn_path):
                print(f"  File tidak ditemukan: {topn_path}")
                for k_eval in range(1, k_max + 1):
                    fold_scores_per_k[k_eval].append(0.0)
                continue

            df_topn = pd.read_csv(topn_path)
            if df_topn.empty:
                for k_eval in range(1, k_max + 1):
                    fold_scores_per_k[k_eval].append(0.0)
                continue

            gt_map = ground_truth_per_fold[fold]
            user_ranked = {
                user_id: group.sort_values("rank")["item_id"].to_numpy()
                for user_id, group in df_topn.groupby("user_id")
            }

            for k_eval in range(1, k_max + 1):
                user_ndcgs = []
                for user_id, ranked_items in user_ranked.items():
                    gt_items = gt_map.get(user_id)
                    if not gt_items:
                        continue

                    dcg = 0.0
                    for rank, item_id in enumerate(
                        ranked_items[:k_eval], start=1
                    ):
                        dcg += (1 / math.log2(rank + 1)) * (
                            1 if item_id in gt_items else 0
                        )

                    idcg_len = min(k_eval, len(gt_items))
                    if idcg_len == 0:
                        continue

                    idcg = 0.0
                    for rank in range(1, idcg_len + 1):
                        idcg += 1 / math.log2(rank + 1)

                    ndcg = dcg / idcg if idcg > 0 else 0.0
                    user_ndcgs.append(
                        {
                            "user_id": user_id,
                            "fold": fold,
                            "k_user": k_user,
                            "k": k_eval,
                            "ndcg": ndcg,
                        }
                    )

                all_user_ndcgs.extend(user_ndcgs)

                fold_ndcg = (
                    float(np.mean([x["ndcg"] for x in user_ndcgs]))
                    if user_ndcgs
                    else 0.0
                )
                fold_scores_per_k[k_eval].append(fold_ndcg)

        # Simpan detail per user ke file CSV di dalam subfolder: ndcg_ib_kr/per_user_ndcg/fold_X/
        if all_user_ndcgs:
            per_user_path = os.path.join(
                per_user_dir, f"k_user_{k_user}.csv"
            )
            pd.DataFrame(all_user_ndcgs).to_csv(
                per_user_path, index=False
            )

        print(f"  k_user={k_user}, topn={k_max}")

        for k_eval in range(1, k_max + 1):
            fold_scores = fold_scores_per_k[k_eval]
            mean_ndcg = (
                float(np.mean(fold_scores)) if fold_scores else 0.0
            )
            results.append(
                {
                    "k_user": k_user,
                    "k": k_eval,
                    "mean_ndcg": mean_ndcg,
                    "fold_scores": fold_scores,
                }
            )

    # Simpan ringkasan hasil evaluasi keseluruhan ke CSV di dalam subfolder: ndcg_hybrid_kr/summary_ndcg.csv
    summary_path = os.path.join(save_dir, "summary_ndcg.csv")
    pd.DataFrame(results).to_csv(summary_path, index=False)

    print(f"Detail per user disimpan di folder: {per_user_dir}")
    print(f"Ringkasan evaluasi disimpan di: {summary_path}")

    return results

def eval_ndcg_from_saved_ubev(
    k_values=None, k_folds=5, k=20, output_dir="hasil_eval_ndcg"
):
    if k_values is None:
        k_values = [10, 20, 30, 40, 50, 100, 150, 200]

    # Menentukan direktori penyimpanan di dalam subfolder 'ndcg_hybrid_kr'
    save_dir = os.path.join(output_dir, "ndcg_ub_ev")
    per_user_dir = os.path.join(save_dir, "per_user_ndcg", f"fold_{k_folds}")
    os.makedirs(per_user_dir, exist_ok=True)

    print("Evaluasi NDCG dari hasil top-N yang tersimpan...")
    results = []

    k_max = int(k)
    if k_max < 1:
        print("k harus >= 1")
        return results

    ground_truth_per_fold = {}
    for fold in range(1, k_folds + 1):
        _, _, df_test, _, _, _, _ = load_data(fold)
        ground_truth_per_fold[fold] = {
            user_id: set(group["item_id"].to_numpy())
            for user_id, group in df_test.groupby("user_id")
        }

    for k_item in k_values:
        fold_scores_per_k = {k_eval: [] for k_eval in range(1, k_max + 1)}
        all_user_ndcgs = []  # Menampung seluruh data per_user untuk pasangan k_user & k_item

        for fold in range(1, k_folds + 1):
            # Tetap membaca top-N yang tersimpan dari folder topn_kr
            topn_path = os.path.join(
                output_dir,
                "topn_ub_EV",
                f"fold_{fold}",
                f"topn_{k_max}_kitem_{k_item}.csv",
            )
            if not os.path.exists(topn_path):
                print(f"  File tidak ditemukan: {topn_path}")
                for k_eval in range(1, k_max + 1):
                    fold_scores_per_k[k_eval].append(0.0)
                continue

            df_topn = pd.read_csv(topn_path)
            if df_topn.empty:
                for k_eval in range(1, k_max + 1):
                    fold_scores_per_k[k_eval].append(0.0)
                continue

            gt_map = ground_truth_per_fold[fold]
            user_ranked = {
                user_id: group.sort_values("rank")["item_id"].to_numpy()
                for user_id, group in df_topn.groupby("user_id")
            }

            for k_eval in range(1, k_max + 1):
                user_ndcgs = []
                for user_id, ranked_items in user_ranked.items():
                    gt_items = gt_map.get(user_id)
                    if not gt_items:
                        continue

                    dcg = 0.0
                    for rank, item_id in enumerate(
                        ranked_items[:k_eval], start=1
                    ):
                        dcg += (1 / math.log2(rank + 1)) * (
                            1 if item_id in gt_items else 0
                        )

                    idcg_len = min(k_eval, len(gt_items))
                    if idcg_len == 0:
                        continue

                    idcg = 0.0
                    for rank in range(1, idcg_len + 1):
                        idcg += 1 / math.log2(rank + 1)

                    ndcg = dcg / idcg if idcg > 0 else 0.0
                    user_ndcgs.append(
                        {
                            "user_id": user_id,
                            "fold": fold,
                            "k_item": k_item,
                            "k": k_eval,
                            "ndcg": ndcg,
                        }
                    )

                all_user_ndcgs.extend(user_ndcgs)

                fold_ndcg = (
                    float(np.mean([x["ndcg"] for x in user_ndcgs]))
                    if user_ndcgs
                    else 0.0
                )
                fold_scores_per_k[k_eval].append(fold_ndcg)

        # Simpan detail per user ke file CSV di dalam subfolder: ndcg_hybrid_kr/per_user_ndcg/fold_X/
        if all_user_ndcgs:
            per_user_path = os.path.join(
                per_user_dir, f"k_item_{k_item}.csv"
            )
            pd.DataFrame(all_user_ndcgs).to_csv(
                per_user_path, index=False
            )

        print(f"  k_item={k_item}, topn={k_max}")

        for k_eval in range(1, k_max + 1):
            fold_scores = fold_scores_per_k[k_eval]
            mean_ndcg = (
                float(np.mean(fold_scores)) if fold_scores else 0.0
            )
            results.append(
                {
                    "k_item": k_item,
                    "k": k_eval,
                    "mean_ndcg": mean_ndcg,
                    "fold_scores": fold_scores,
                }
            )

    # Simpan ringkasan hasil evaluasi keseluruhan ke CSV di dalam subfolder: ndcg_hybrid_kr/summary_ndcg.csv
    summary_path = os.path.join(save_dir, "summary_ndcg.csv")
    pd.DataFrame(results).to_csv(summary_path, index=False)

    print(f"Detail per user disimpan di folder: {per_user_dir}")
    print(f"Ringkasan evaluasi disimpan di: {summary_path}")

    return results

def eval_ndcg_from_saved_ibev(
    k_values=None, k_folds=5, k=20, output_dir="hasil_eval_ndcg"
):
    if k_values is None:
        k_values = [10, 20, 30, 40, 50, 100, 150, 200]

    # Menentukan direktori penyimpanan di dalam subfolder 'ndcg_hybrid_kr'
    save_dir = os.path.join(output_dir, "ndcg_ib_ev")
    per_user_dir = os.path.join(save_dir, "per_user_ndcg", f"fold_{k_folds}")
    os.makedirs(per_user_dir, exist_ok=True)

    print("Evaluasi NDCG dari hasil top-N yang tersimpan...")
    results = []

    k_max = int(k)
    if k_max < 1:
        print("k harus >= 1")
        return results

    ground_truth_per_fold = {}
    for fold in range(1, k_folds + 1):
        _, _, df_test, _, _, _, _ = load_data(fold)
        ground_truth_per_fold[fold] = {
            user_id: set(group["item_id"].to_numpy())
            for user_id, group in df_test.groupby("user_id")
        }

    for k_user in k_values:
        fold_scores_per_k = {k_eval: [] for k_eval in range(1, k_max + 1)}
        all_user_ndcgs = []  # Menampung seluruh data per_user untuk pasangan k_user & k_item

        for fold in range(1, k_folds + 1):
            # Tetap membaca top-N yang tersimpan dari folder topn_kr
            topn_path = os.path.join(
                output_dir,
                "topn_ib_EV",
                f"fold_{fold}",
                f"topn_{k_max}_kuser_{k_user}.csv",
            )
            if not os.path.exists(topn_path):
                print(f"  File tidak ditemukan: {topn_path}")
                for k_eval in range(1, k_max + 1):
                    fold_scores_per_k[k_eval].append(0.0)
                continue

            df_topn = pd.read_csv(topn_path)
            if df_topn.empty:
                for k_eval in range(1, k_max + 1):
                    fold_scores_per_k[k_eval].append(0.0)
                continue

            gt_map = ground_truth_per_fold[fold]
            user_ranked = {
                user_id: group.sort_values("rank")["item_id"].to_numpy()
                for user_id, group in df_topn.groupby("user_id")
            }

            for k_eval in range(1, k_max + 1):
                user_ndcgs = []
                for user_id, ranked_items in user_ranked.items():
                    gt_items = gt_map.get(user_id)
                    if not gt_items:
                        continue

                    dcg = 0.0
                    for rank, item_id in enumerate(
                        ranked_items[:k_eval], start=1
                    ):
                        dcg += (1 / math.log2(rank + 1)) * (
                            1 if item_id in gt_items else 0
                        )

                    idcg_len = min(k_eval, len(gt_items))
                    if idcg_len == 0:
                        continue

                    idcg = 0.0
                    for rank in range(1, idcg_len + 1):
                        idcg += 1 / math.log2(rank + 1)

                    ndcg = dcg / idcg if idcg > 0 else 0.0
                    user_ndcgs.append(
                        {
                            "user_id": user_id,
                            "fold": fold,
                            "k_user": k_user,
                            "k": k_eval,
                            "ndcg": ndcg,
                        }
                    )

                all_user_ndcgs.extend(user_ndcgs)

                fold_ndcg = (
                    float(np.mean([x["ndcg"] for x in user_ndcgs]))
                    if user_ndcgs
                    else 0.0
                )
                fold_scores_per_k[k_eval].append(fold_ndcg)

        # Simpan detail per user ke file CSV di dalam subfolder: ndcg_hybrid_kr/per_user_ndcg/fold_X/
        if all_user_ndcgs:
            per_user_path = os.path.join(
                per_user_dir, f"k_user_{k_user}.csv"
            )
            pd.DataFrame(all_user_ndcgs).to_csv(
                per_user_path, index=False
            )

        print(f"  k_user={k_user}, topn={k_max}")

        for k_eval in range(1, k_max + 1):
            fold_scores = fold_scores_per_k[k_eval]
            mean_ndcg = (
                float(np.mean(fold_scores)) if fold_scores else 0.0
            )
            results.append(
                {
                    "k_user": k_user,
                    "k": k_eval,
                    "mean_ndcg": mean_ndcg,
                    "fold_scores": fold_scores,
                }
            )

    # Simpan ringkasan hasil evaluasi keseluruhan ke CSV di dalam subfolder: ndcg_hybrid_kr/summary_ndcg.csv
    summary_path = os.path.join(save_dir, "summary_ndcg.csv")
    pd.DataFrame(results).to_csv(summary_path, index=False)

    print(f"Detail per user disimpan di folder: {per_user_dir}")
    print(f"Ringkasan evaluasi disimpan di: {summary_path}")

    return results

In [51]:
run_cv_5fold_grid(n_jobs=15) 

Inisialisasi uji 5-fold CV (staged, n_jobs=15, precompute_jobs=15)...
Tahap A: precompute prior dan ITR per fold...


[Parallel(n_jobs=15)]: Using backend LokyBackend with 15 concurrent workers.
[Parallel(n_jobs=15)]: Done   2 out of   5 | elapsed:    1.3s remaining:    2.0s
[Parallel(n_jobs=15)]: Done   3 out of   5 | elapsed:    1.3s remaining:    0.8s
[Parallel(n_jobs=15)]: Done   5 out of   5 | elapsed:    1.3s finished
[Parallel(n_jobs=15)]: Using backend LokyBackend with 15 concurrent workers.


Tahap B1: Likelihood parallel untuk semua kombinasi...


[Parallel(n_jobs=15)]: Done   2 tasks      | elapsed:    1.3s
[Parallel(n_jobs=15)]: Done  11 tasks      | elapsed:    1.3s
[Parallel(n_jobs=15)]: Done  20 tasks      | elapsed:    1.6s
[Parallel(n_jobs=15)]: Done  31 tasks      | elapsed:    1.8s
[Parallel(n_jobs=15)]: Done  42 tasks      | elapsed:    1.9s
[Parallel(n_jobs=15)]: Done  55 tasks      | elapsed:    2.2s
[Parallel(n_jobs=15)]: Done  68 tasks      | elapsed:    2.7s
[Parallel(n_jobs=15)]: Done  83 tasks      | elapsed:    2.9s
[Parallel(n_jobs=15)]: Done  98 tasks      | elapsed:    3.2s
[Parallel(n_jobs=15)]: Done 115 tasks      | elapsed:    3.5s
[Parallel(n_jobs=15)]: Done 132 tasks      | elapsed:    4.1s
[Parallel(n_jobs=15)]: Done 151 tasks      | elapsed:    4.3s
[Parallel(n_jobs=15)]: Done 170 tasks      | elapsed:    4.6s
[Parallel(n_jobs=15)]: Done 191 tasks      | elapsed:    5.1s
[Parallel(n_jobs=15)]: Done 212 tasks      | elapsed:    5.8s
[Parallel(n_jobs=15)]: Done 235 tasks      | elapsed:    6.1s
[Paralle

Tahap B2: Posterior parallel untuk semua kombinasi...


...

[Parallel(n_jobs=15)]: Done   2 tasks      | elapsed:    1.8s
[Parallel(n_jobs=15)]: Done  11 tasks      | elapsed:    2.1s
[Parallel(n_jobs=15)]: Done  20 tasks      | elapsed:    3.5s
[Parallel(n_jobs=15)]: Done  31 tasks      | elapsed:    4.2s
[Parallel(n_jobs=15)]: Done  42 tasks      | elapsed:    5.7s
[Parallel(n_jobs=15)]: Done  55 tasks      | elapsed:    7.4s
[Parallel(n_jobs=15)]: Done  68 tasks      | elapsed:    9.4s
[Parallel(n_jobs=15)]: Done  83 tasks      | elapsed:   11.9s
[Parallel(n_jobs=15)]: Done  98 tasks      | elapsed:   14.0s
[Parallel(n_jobs=15)]: Done 115 tasks      | elapsed:   16.4s
[Parallel(n_jobs=15)]: Done 132 tasks      | elapsed:   19.8s
[Parallel(n_jobs=15)]: Done 151 tasks      | elapsed:   23.4s
[Parallel(n_jobs=15)]: Done 170 tasks      | elapsed:   26.7s
[Parallel(n_jobs=15)]: Done 191 tasks      | elapsed:   31.6s
[Parallel(n_jobs=15)]: Done 212 tasks      | elapsed:   36.2s
[Parallel(n_jobs=15)]: Done 235 tasks      | elapsed:   40.4s
[Paralle

Tahap B4F: TopN parallel untuk semua kombinasi...


[Parallel(n_jobs=15)]: Done   2 tasks      | elapsed:    2.8s
[Parallel(n_jobs=15)]: Done  11 tasks      | elapsed:    3.1s
[Parallel(n_jobs=15)]: Done  20 tasks      | elapsed:    5.1s
[Parallel(n_jobs=15)]: Done  31 tasks      | elapsed:    5.9s
[Parallel(n_jobs=15)]: Done  42 tasks      | elapsed:    7.8s
[Parallel(n_jobs=15)]: Done  55 tasks      | elapsed:    9.5s
[Parallel(n_jobs=15)]: Done  68 tasks      | elapsed:   12.3s
[Parallel(n_jobs=15)]: Done  83 tasks      | elapsed:   15.1s
[Parallel(n_jobs=15)]: Done  98 tasks      | elapsed:   17.5s
[Parallel(n_jobs=15)]: Done 115 tasks      | elapsed:   20.3s
[Parallel(n_jobs=15)]: Done 132 tasks      | elapsed:   24.2s
[Parallel(n_jobs=15)]: Done 151 tasks      | elapsed:   28.3s
[Parallel(n_jobs=15)]: Done 170 tasks      | elapsed:   32.4s
[Parallel(n_jobs=15)]: Done 191 tasks      | elapsed:   37.0s
[Parallel(n_jobs=15)]: Done 212 tasks      | elapsed:   42.9s
[Parallel(n_jobs=15)]: Done 235 tasks      | elapsed:   47.7s
[Paralle

Selesai CV staged.


[Parallel(n_jobs=15)]: Done 320 out of 320 | elapsed:  1.1min finished


In [ ]:
eval_ndcg_from_saved_hbkr(k=100, output_dir="hasil_eval_ndcg")

Evaluasi NDCG dari hasil top-N yang tersimpan...
  k_user=10, k_item=10, topn=100
  k_user=10, k_item=20, topn=100
  k_user=10, k_item=30, topn=100
  k_user=10, k_item=40, topn=100
  k_user=10, k_item=50, topn=100
  k_user=10, k_item=100, topn=100
  k_user=10, k_item=150, topn=100
  k_user=10, k_item=200, topn=100
  k_user=20, k_item=10, topn=100
  k_user=20, k_item=20, topn=100
  k_user=20, k_item=30, topn=100
  k_user=20, k_item=40, topn=100
  k_user=20, k_item=50, topn=100
  k_user=20, k_item=100, topn=100
  k_user=20, k_item=150, topn=100
  k_user=20, k_item=200, topn=100
  k_user=30, k_item=10, topn=100
  k_user=30, k_item=20, topn=100
  k_user=30, k_item=30, topn=100
  k_user=30, k_item=40, topn=100
  k_user=30, k_item=50, topn=100
  k_user=30, k_item=100, topn=100


KeyboardInterrupt: 

In [ ]:
eval_ndcg_from_saved_hbev(k=100, output_dir="hasil_eval_ndcg")

Evaluasi NDCG dari hasil top-N yang tersimpan...
  k_user=10, k_item=10, topn=100
  k_user=10, k_item=20, topn=100
  k_user=10, k_item=30, topn=100
  k_user=10, k_item=40, topn=100
  k_user=10, k_item=50, topn=100
  k_user=10, k_item=100, topn=100
  k_user=10, k_item=150, topn=100
  k_user=10, k_item=200, topn=100
  k_user=20, k_item=10, topn=100
  k_user=20, k_item=20, topn=100
  k_user=20, k_item=30, topn=100
  k_user=20, k_item=40, topn=100
  k_user=20, k_item=50, topn=100
  k_user=20, k_item=100, topn=100
  k_user=20, k_item=150, topn=100
  k_user=20, k_item=200, topn=100
  k_user=30, k_item=10, topn=100
  k_user=30, k_item=20, topn=100
  k_user=30, k_item=30, topn=100
  k_user=30, k_item=40, topn=100
  k_user=30, k_item=50, topn=100
  k_user=30, k_item=100, topn=100
  k_user=30, k_item=150, topn=100
  k_user=30, k_item=200, topn=100
  k_user=40, k_item=10, topn=100
  k_user=40, k_item=20, topn=100
  k_user=40, k_item=30, topn=100
  k_user=40, k_item=40, topn=100
  k_user=40, k_ite

In [52]:
eval_ndcg_from_saved_ubkr(k=100, output_dir="hasil_eval_ndcg")

Evaluasi NDCG dari hasil top-N yang tersimpan...
  k_item=10, topn=100
  k_item=20, topn=100
  k_item=30, topn=100
  k_item=40, topn=100
  k_item=50, topn=100
  k_item=100, topn=100
  k_item=150, topn=100
  k_item=200, topn=100
Detail per user disimpan di folder: hasil_eval_ndcg\ndcg_ub_kr\per_user_ndcg\fold_5
Ringkasan evaluasi disimpan di: hasil_eval_ndcg\ndcg_ub_kr\summary_ndcg.csv


[{'k_item': 10,
  'k': 1,
  'mean_ndcg': 0.07870222931329413,
  'fold_scores': [0.0915032679738562,
   0.10260336906584992,
   0.07940161104718067,
   0.06067172264355363,
   0.0593311758360302]},
 {'k_item': 10,
  'k': 2,
  'mean_ndcg': 0.07023586379144156,
  'fold_scores': [0.08476073538589034,
   0.08957004324784086,
   0.06604650838085586,
   0.056061342492329407,
   0.054740689450291306]},
 {'k_item': 10,
  'k': 3,
  'mean_ndcg': 0.0689662148922158,
  'fold_scores': [0.08736519267602577,
   0.08795694676815166,
   0.06350991489963763,
   0.05285476823635893,
   0.05314425188090497]},
 {'k_item': 10,
  'k': 4,
  'mean_ndcg': 0.06680593555405982,
  'fold_scores': [0.0859491177787688,
   0.08346773911705223,
   0.06273834519011547,
   0.05124896403938099,
   0.05062551164498165]},
 {'k_item': 10,
  'k': 5,
  'mean_ndcg': 0.0658097442503945,
  'fold_scores': [0.08645907226841695,
   0.08163648320866124,
   0.060845498654845655,
   0.050830595393609984,
   0.04927707172643872]},
 {'k_i

In [53]:
eval_ndcg_from_saved_ubev(k=100, output_dir="hasil_eval_ndcg")

Evaluasi NDCG dari hasil top-N yang tersimpan...
  k_item=10, topn=100
  k_item=20, topn=100
  k_item=30, topn=100
  k_item=40, topn=100
  k_item=50, topn=100
  k_item=100, topn=100
  k_item=150, topn=100
  k_item=200, topn=100
Detail per user disimpan di folder: hasil_eval_ndcg\ndcg_ub_ev\per_user_ndcg\fold_5
Ringkasan evaluasi disimpan di: hasil_eval_ndcg\ndcg_ub_ev\summary_ndcg.csv


[{'k_item': 10,
  'k': 1,
  'mean_ndcg': 0.1327218413908366,
  'fold_scores': [0.1721132897603486,
   0.1562021439509954,
   0.1093210586881473,
   0.11917659804983749,
   0.10679611650485436]},
 {'k_item': 10,
  'k': 2,
  'mean_ndcg': 0.12534348135973344,
  'fold_scores': [0.16115667430490405,
   0.14553851373626073,
   0.10575969797712734,
   0.11330884149373392,
   0.10095367928664123]},
 {'k_item': 10,
  'k': 3,
  'mean_ndcg': 0.11941257368503191,
  'fold_scores': [0.1555483776672368,
   0.1386981656421418,
   0.10360353861478777,
   0.10662874091385172,
   0.09258404558714135]},
 {'k_item': 10,
  'k': 4,
  'mean_ndcg': 0.11534756062979871,
  'fold_scores': [0.15474451632669273,
   0.13373574617241965,
   0.09910626619640352,
   0.10118652453279656,
   0.08796474992068119]},
 {'k_item': 10,
  'k': 5,
  'mean_ndcg': 0.11300169121039427,
  'fold_scores': [0.15055467280639986,
   0.12963751078960078,
   0.09919414166741092,
   0.10095618404965197,
   0.08466594673890791]},
 {'k_item':

In [54]:
eval_ndcg_from_saved_ibkr(k=100, output_dir="hasil_eval_ndcg")

Evaluasi NDCG dari hasil top-N yang tersimpan...
  k_user=10, topn=100
  k_user=20, topn=100
  k_user=30, topn=100
  k_user=40, topn=100
  k_user=50, topn=100
  k_user=100, topn=100
  k_user=150, topn=100
  k_user=200, topn=100
Detail per user disimpan di folder: hasil_eval_ndcg\ndcg_ib_kr\per_user_ndcg\fold_5
Ringkasan evaluasi disimpan di: hasil_eval_ndcg\ndcg_ib_kr\summary_ndcg.csv


[{'k_user': 10,
  'k': 1,
  'mean_ndcg': 0.09720266348070408,
  'fold_scores': [0.12854030501089325,
   0.11791730474732007,
   0.08170310701956271,
   0.08234019501625135,
   0.07551240560949299]},
 {'k_user': 10,
  'k': 2,
  'mean_ndcg': 0.09042495926928108,
  'fold_scores': [0.12348340556991884,
   0.1060688267309482,
   0.07769657621966528,
   0.0731194347138029,
   0.07175655311207024]},
 {'k_user': 10,
  'k': 3,
  'mean_ndcg': 0.08657347173304668,
  'fold_scores': [0.12109157514842082,
   0.1013031166270195,
   0.071076364369377,
   0.07146978062143043,
   0.06792652189898565]},
 {'k_user': 10,
  'k': 4,
  'mean_ndcg': 0.08345679129586431,
  'fold_scores': [0.11545841419098357,
   0.09894703058935067,
   0.07054134334429298,
   0.06955765708244893,
   0.06277951127224539]},
 {'k_user': 10,
  'k': 5,
  'mean_ndcg': 0.08131709126571167,
  'fold_scores': [0.10991407558209669,
   0.09596695723150651,
   0.06992151827690749,
   0.06908258116453043,
   0.0617003240735173]},
 {'k_user':

In [55]:
eval_ndcg_from_saved_ibev(k=100, output_dir="hasil_eval_ndcg")

Evaluasi NDCG dari hasil top-N yang tersimpan...
  k_user=10, topn=100
  k_user=20, topn=100
  k_user=30, topn=100
  k_user=40, topn=100
  k_user=50, topn=100
  k_user=100, topn=100
  k_user=150, topn=100
  k_user=200, topn=100
Detail per user disimpan di folder: hasil_eval_ndcg\ndcg_ib_ev\per_user_ndcg\fold_5
Ringkasan evaluasi disimpan di: hasil_eval_ndcg\ndcg_ib_ev\summary_ndcg.csv


[{'k_user': 10,
  'k': 1,
  'mean_ndcg': 0.08614809635050324,
  'fold_scores': [0.12200435729847495,
   0.09341500765696784,
   0.07479861910241657,
   0.06608884073672806,
   0.0744336569579288]},
 {'k_user': 10,
  'k': 2,
  'mean_ndcg': 0.08309459796806633,
  'fold_scores': [0.11779027443099628,
   0.09104531205369347,
   0.07613412936904905,
   0.06525058980014184,
   0.06525268418645101]},
 {'k_user': 10,
  'k': 3,
  'mean_ndcg': 0.0814136152993245,
  'fold_scores': [0.11673427667347527,
   0.09052336028652742,
   0.0726850912232793,
   0.06427455066010838,
   0.06285079765323211]},
 {'k_user': 10,
  'k': 4,
  'mean_ndcg': 0.08040142861464712,
  'fold_scores': [0.1157888677014304,
   0.08972217869631446,
   0.07214791668976733,
   0.062185383189888525,
   0.06216279679583486]},
 {'k_user': 10,
  'k': 5,
  'mean_ndcg': 0.08006735859899192,
  'fold_scores': [0.11351581827078448,
   0.09129116223004796,
   0.07197297650922263,
   0.0617034984219013,
   0.061853337563003236]},
 {'k_use